## XGBoost: Regression vs Classification

### Regression

**Similarity Score**

$$
\text{Similarity Score} = \frac{\left( \sum \text{Residuals} \right)^2}{N + \lambda}
$$

**Output Value (Leaf Weight)**

$$
\text{Output Value} = \frac{\sum \text{Residuals}}{N + \lambda}
$$

---

### Classification

**Similarity Score**

$$
\text{Similarity Score} = \frac{\left( \sum \text{Residuals} \right)^2}{\sum \left[ P(1 - P) \right] + \lambda}
$$

**Output Value (Leaf Weight)**

$$
\text{Output Value} = \frac{\sum \text{Residuals}}{\sum \left[ P(1 - P) \right] + \lambda}
$$

---

## Key Difference

- In regression, the denominator depends on:

$$
N
$$

- In classification, the denominator depends on:

$$
P(1 - P)
$$

- The term $P(1 - P)$ comes from the second derivative (Hessian) of the log loss.

----
---

## Why XGBoost Uses a Different Tree

In a standard Decision Tree, we use criteria like entropy or Gini impurity to decide splits.

For example, in classification:

$$
\text{Entropy} = - \sum p_i \log(p_i)
$$

However, XGBoost does not use entropy or Gini.

Instead, it builds trees using **gradient boosting**, where each split is evaluated using a custom objective function.

---

## Key Idea in XGBoost

- Instead of impurity, XGBoost uses a quantity called **Similarity Score**
- This comes from optimizing a loss function using:
  - First derivative (Gradient)
  - Second derivative (Hessian)

---

## Difference in Output Value (Leaf Value)

In a normal Decision Tree:

- Regression → output is mean of target values
- Classification → output is majority class

There is no optimization happening at the leaf level.

---

## XGBoost Output Value

In XGBoost, each leaf stores a value that minimizes the loss function.

The formula is:

$$
\text{Output Value} = \frac{\sum \text{Residuals}}{\text{Denominator} + \lambda}
$$

Where:

- Residuals = gradients
- Denominator =
  - Regression: $N$
  - Classification: $\sum P(1 - P)$
- $\lambda$ = regularization term

---

## What We Will Derive

In this section, we will derive:

1. The Output Value (leaf weight)
2. The Similarity Score used for splitting

These are obtained by minimizing the regularized objective function in XGBoost.

---
---

# Mathematics of XGBoost 
[referOriginalPaper](https://arxiv.org/pdf/1603.02754)

## Prerequisites

Before diving into the mathematical derivation of XGBoost, make sure you are familiar with the following concepts:

- XGBoost (Regression & Classification)  
  Refer to previous notebooks for implementation and intuition.

- Gradient Boosting  
  Understanding how models are built sequentially using residuals.

- Tree Bias  
  Understanding how decision trees make predictions and their inherent biases.

---

# Quick Revision

Gradient boosting models are **additive models**, where multiple weak learners are combined in an ensemble manner.

- We start by training an initial model.
- The combined model makes some errors.
- A new model is then trained to correct these errors (residuals).
- This new model is added to the existing ensemble.

The process continues iteratively, with each new model trying to reduce the residuals of the combined model.

The goal is to progressively minimize the error, ideally driving the residuals toward zero.

## Dataset

| CGPA | IQ  | Package |
|------|-----|---------|
| 6.5  | 71  | 13      |
| 5.0  | 91  | 10      |
| 9.1  | 120 | 16      |

## Gradient Boosting in Regression

In regression, the first model predicts a constant value, which is the **average of the target variable**:

$$
f_1(x) = \bar{y}
$$

---

### Stage 2

- Compute residuals:

$$
r_i = y_i - f_1(x_i)
$$

- Train a new decision tree:
  - Input: features (e.g., CGPA)
  - Output: residuals

This gives the second model:

$$
f_2(x)
$$

---

### Prediction (2 Models)

If we train only two models, the prediction for a new data point is:

$$
\hat{y} = f_1(x) + f_2(x)
$$

---

## General Additive Model

Gradient boosting models are **additive models**:

$$
f(x) = f_1(x) + f_2(x) + \dots + f_n(x)
$$

$$
\hat{y}_i = \sum_{k=1}^{n} f_k(x_i)
$$

---

## XGBoost Formulation (Regression)

In XGBoost, each decision tree outputs a value at the leaf node (called **leaf weight**).

For regression, the output of a leaf is:

$$
w = \frac{\sum r_i}{N + \lambda}
$$

Where:
- $r_i$ = residuals
- $N$ = number of samples in the leaf
- $\lambda$ = regularization parameter

---

## Final Prediction with Learning Rate

XGBoost introduces a learning rate $\eta$ to control the contribution of each tree:

$$
\hat{y} = f_1(x) + \eta f_2(x) + \eta f_3(x) + \dots
$$

---

## Objective of This Notebook

In this notebook, we will focus on deriving:

and understanding how this formula arises from the optimization of the XGBoost objective function.

---------
-------

# XGBoost loss function

XGBoost builds an additive model where predictions are made as a sum of functions:

$$
\hat{y}_i = \sum_{k=1}^{n} f_k(x_i)
$$

Each function $f_k$ represents a decision tree, and each tree consists of leaf nodes that output constant values.

The total prediction is obtained by summing the outputs of all trees.

In reality, we have true values $y_i$, and our goal is to minimize the difference between $y_i$ and $\hat{y}_i$.

For simplicity, consider only two functions:

$$
\hat{y}_i = f_1(x_i) + f_2(x_i)
$$

For every data point, the second function $f_2(x_i)$ is trained to reduce the difference between $y_i$ and $\hat{y}_i$.

To measure this difference, we define a loss function:

$$
L(y_i, \hat{y}_i)
$$

Our objective is to minimize this loss.

In regression, a commonly used loss function is Mean Squared Error:

$$
L(y_i, \hat{y}_i) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

However, in gradient boosting, the framework is flexible and allows the use of any loss function, as long as it is differentiable.

$$
L(y_i, \hat{y}_i)
$$

This means we can use different loss functions such as Mean Squared Error or log loss (log-odds), depending on the problem.

In XGBoost, the formulation is slightly different, as it introduces regularization and uses both first and second order derivatives of the loss function.

---

We start with an additive model:

$$
\hat{y} = m_1 + m_2
$$

Where:
- $m_1$ is the base model  
- $m_2$ is the new tree added at the current step $t$

---

### The objective function in XGBoost is defined as:

$$
L = \sum_{i=1}^{n} L(y_i, \hat{y}_i) + \Omega(f_2(x_i))
$$

Where:
- $L(y_i, \hat{y}_i)$ is the loss function
- $\Omega(f_2)$ is the regularization term for the new tree

The regularization term in XGBoost is defined as:

$$
\Omega(f) = \gamma T + \frac{1}{2} \lambda \|w\|^2
$$

Where:

- $T$ = number of leaf nodes in the tree  
- $\gamma$ = penalty for adding a new leaf (controls model complexity)  
- $\lambda$ = L2 regularization term on leaf weights  
- $w$ = vector of leaf weights  

This term helps to control overfitting by penalizing:
- large number of leaves  
- large leaf weights

In XGBoost, 

$$
\Omega(f)
$$

is the extra regularization term added to the objective function.

Our goal is to find the values of the leaf weights $w$ such that the overall loss function is minimized.

This means we optimize both:
- the training loss
- the regularization term

to obtain the best possible model.

# The derivation

The objective function in XGBoost is defined as:

$$
\mathcal{L} = \sum_{i=1}^{n} L(y_i, \hat{y}_i) + \Omega(f_k)
$$

Where:

- $\sum_{i=1}^{n} L(y_i, \hat{y}_i)$ represents the loss function  
- $\Omega(f_k)$ represents the regularization term for the $k$-th tree  

The first term measures how well the model fits the data, while the second term controls the complexity of the model.

## Getting Familiar with the Objective Function

In boosting, models are added sequentially, and at each stage we optimize an objective function.

Stage 1:

If we use only one model, the objective function is:

$$
\mathcal{L}^{(1)} = \sum_{i=1}^{n} L(y_i, \hat{y}_i)
$$

---

Stage 2:

Now we introduce a second model:

$$
\hat{y}_i = f_1(x_i) + f_2(x_i)
$$

The objective function becomes:

$$
\mathcal{L}^{(2)} = \sum_{i=1}^{n} L\left(y_i, f_1(x_i) + f_2(x_i)\right) + \Omega(f_2)
$$

Here, the regularization term is applied only to the newly added model $f_2$.

---

Stage 3:

Now we add a third model:

$$
\hat{y}_i = f_1(x_i) + f_2(x_i) + f_3(x_i)
$$

The objective function becomes:

$$
\mathcal{L}^{(3)} = \sum_{i=1}^{n} L\left(y_i, f_1(x_i) + f_2(x_i) + f_3(x_i)\right) + \Omega(f_3)
$$

Only $\Omega(f_3)$ is added because the regularization for previous models has already been accounted for in earlier stages.

---

Stage $t$:

At stage $t$, we have:

$$
\hat{y}_i = \sum_{k=1}^{t} f_k(x_i)
$$

The objective function becomes:

$$
\mathcal{L}^{(t)} = \sum_{i=1}^{n} L\left(y_i, \sum_{k=1}^{t} f_k(x_i)\right) + \Omega(f_t)
$$

---

We can simplify the prediction term:

$$
\hat{y}_i^{(t-1)} = \sum_{k=1}^{t-1} f_k(x_i)
$$

So the objective function can be rewritten as:

$$
\mathcal{L}^{(t)} = \sum_{i=1}^{n} L\left(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\right) + \Omega(f_t)
$$

At any stage $t$, this is the form of the objective function we optimize.

---

## [Reference: Gradient Tree Boosting (Section 2.2)](https://arxiv.org/pdf/1603.02754)


---
---

## Taylor Series Expansion

$$
\mathcal{L}^{(t)} = \sum_{i=1}^{n} L\left(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\right) + \Omega(f_t)
$$

At this stage, the objective is to optimize the function. We want to find the value of $f_t(x_i)$ such that the objective function is minimized.

In linear regression, we have already solved a similar optimization problem.

For example, the loss function is Mean Squared Error:

$$
\sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

Where:

$$
\hat{y}_i = m x_i + b
$$

So the loss becomes:

$$
L = \sum_{i=1}^{n} (y_i - m x_i - b)^2
$$

If we visualize this, we can think of a 3D surface with axes $m$, $b$, and $L$.  
This surface forms a paraboloid, and the minimum occurs where:

$$
\frac{\partial L}{\partial m} = 0 \quad \text{and} \quad \frac{\partial L}{\partial b} = 0
$$

This gives the optimal values of $m$ and $b$.

In short, we already know how to solve such optimization problems when the model is simple.

However, in XGBoost, the situation is more complex.

Here, the prediction is:

$$
\hat{y}_i = \hat{y}_i^{(t-1)} + f_t(x_i)
$$

Where:
- $\hat{y}_i^{(t-1)}$ is constant (from previous trees)
- $f_t(x_i)$ is a decision tree

Unlike linear regression, where the model is a smooth function ($mx + b$), a decision tree produces a piecewise constant function.
[DecisionTreeRegressionGraph](https://www.google.com/search?sca_esv=ab78fc3432efc4ca&sxsrf=ANbL-n61DDewB0GpCx0qi1DLfo_9Q_gqtA:1776426808988&udm=2&fbs=ADc_l-aN0CWEZBOHjofHoaMMDiKpaEWjvZ2Py1XXV8d8KvlI3p-ML-906rRL_m6h4jR-tdCeKIwp94h-QiJ4lJfObsqUPixp6KuAej6LdEw-ul8fudU82HJrKxYOXZophBT2wsL1A8SvxreDpxGmVqCpVkesfdCEYv4TQqqnLiFMWwhJ20FI1ggBlnQwmM6ujC1A3bTpl2yDZ6XL120IwEWHvfjJYiiIUg&q=decision+tree+regression+graph&sa=X&ved=2ahUKEwjQ-qzH6fSTAxWWRmwGHc8mIukQtKgLegQIExAB&biw=920&bih=695&dpr=1.25#sv=CAMSVhoyKhBlLXdydVBVY1hwSTJxcWFNMg53cnVQVWNYcEkycXFhTToOR1dscWYzdlcwekRhOE0gBCocCgZtb3NhaWMSEGUtd3J1UFVjWHBJMnFxYU0YADABGAcg0s2BO0oIEAIYASACKAE)


This means the prediction function looks like a step-wise curve instead of a smooth line.

As a result, if we try to visualize the loss surface, it will not be a smooth parabola.  
Instead, it can have irregular shapes and sharp corners.

At many points, the function is not differentiable, so we cannot directly compute gradients in the usual way.

In summary, we want to find $f_t(x_i)$ such that:

$$
\frac{\partial \mathcal{L}^{(t)}}{\partial f_t(x_i)} = 0
$$

But since $\hat{y}_i$ depends on a decision tree (a non-smooth function), directly computing this derivative is not straightforward.

This is why we use Taylor Series Expansion to approximate the objective function and make optimization possible.

# Solution: Taylor Series

### What?

Taylor Series is a powerful approximation technique.

It allows us to approximate a complex function using a polynomial.

$$
\text{complex function} \;\;\rightarrow\;\; \text{polynomial approximation}
$$

For example, a function like:

$$
e^x
$$

can be approximated using a polynomial (not exact, just an approximation):

$$
e^x \approx 3x^2 + 2
$$

In reality, Taylor Series expands as:

$$
e^x = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + \dots
$$

To achieve higher accuracy, we need to include more polynomial terms.

In practice, we often limit the expansion to a few terms for computational efficiency.

### [Watch-Taylor Series In Depth](https://www.youtube.com/watch?v=3d6DsjIBzJ4)

### How?

Taylor Series uses derivatives to approximate a function around a point $a$.

The general form is:

$$
f(x) \approx f(a) + \frac{f'(a)}{1!}(x - a) + \frac{f''(a)}{2!}(x - a)^2 + \frac{f'''(a)}{3!}(x - a)^3 + \dots
$$

---

Let:

$$
f(x) = e^x
$$

We want to approximate this function around $a = 0$.

First, evaluate the function and its derivatives at $a = 0$:

$$
f(a) = e^a = e^0 = 1
$$

$$
f'(a) = e^a = 1
\quad \Rightarrow \quad \frac{f'(a)}{1!}(x - a) = x
$$

$$
f''(a) = e^a = 1
\quad \Rightarrow \quad \frac{f''(a)}{2!}(x - a)^2 = \frac{x^2}{2}
$$

$$
f'''(a) = e^a = 1
\quad \Rightarrow \quad \frac{f'''(a)}{3!}(x - a)^3 = \frac{x^3}{6}
$$

---

Putting everything together:

$$
e^x = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + \dots
$$

This is the Taylor Series expansion of $e^x$ around $a = 0$.

> ## [visualizeUsingDesmos](https://www.desmos.com/calculator)



---
---

# Applying taylor series 


Using Taylor Series expansion up to the second order:

$$
f(x) \approx f(a) + f'(a)(x - a) + \frac{f''(a)}{2!}(x - a)^2
$$

We apply this idea to the XGBoost objective function at stage $t$:

$$
\mathcal{L}^{(t)} = \sum_{i=1}^{n} L\left(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\right) + \Omega(f_t)
$$

Using second-order Taylor expansion, we approximate the loss function.

This is why XGBoost uses a **second-order approximation** to make optimization tractable.

------

To apply Taylor Series, we define:

$$
x = \hat{y}_i^{(t-1)} + f_t(x_i)
$$

$$
a = \hat{y}_i^{(t-1)}
$$

This means we are approximating the loss function around the current prediction $\hat{y}_i^{(t-1)}$.

So, the Taylor expansion is performed with respect to the new function $f_t(x_i)$, treating the previous prediction as a constant reference point.

---
---

Applying Taylor expansion terms to the objective function:

First term:

$$
f(a) = \sum_{i=1}^{n} L\left(y_i, \hat{y}_i^{(t-1)}\right)
$$

---

Second term:

$$
f'(a)(x - a) = \sum_{i=1}^{n} \frac{\partial L\left(y_i, \hat{y}_i^{(t-1)}\right)}{\partial \hat{y}_i^{(t-1)}} \; f_t(x_i)
$$

---

Third term:

$$
\frac{f''(a)}{2}(x - a)^2 = \sum_{i=1}^{n} \frac{1}{2} \frac{\partial^2 L\left(y_i, \hat{y}_i^{(t-1)}\right)}{\partial \left(\hat{y}_i^{(t-1)}\right)^2} \; f_t^2(x_i)
$$

---
# Simiplification 
---

After applying the second-order Taylor expansion, the objective function can be approximated as:

$$
\mathcal{L}^{(t)} \approx \sum_{i=1}^{n} \left[ L\left(y_i, \hat{y}_i^{(t-1)}\right) + g_i \, f_t(x_i) + \frac{1}{2} h_i \, f_t^2(x_i) \right] + \Omega(f_t)                       
$$

Where:

$$
g_i = \frac{\partial L\left(y_i, \hat{y}_i^{(t-1)}\right)}{\partial \hat{y}_i^{(t-1)}}
$$

is the **first derivative (gradient)** of the loss function.

$$
h_i = \frac{\partial^2 L\left(y_i, \hat{y}_i^{(t-1)}\right)}{\partial \left(\hat{y}_i^{(t-1)}\right)^2}
$$

is the **second derivative (Hessian)** of the loss function.

These terms allow XGBoost to use both first-order and second-order information for optimization.

------
-------

We start with the simplified objective function:

$$
\tilde{\mathcal{L}}^{(t)} = \sum_{i=1}^{n} \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \Omega(f_t)
$$

Originally, the full expression also contained the term:

$$
\sum_{i=1}^{n} L\left(y_i, \hat{y}_i^{(t-1)}\right)
$$

However, this term does not depend on $f_t(x_i)$.

Since we are optimizing with respect to $f_t(x_i)$, this term becomes a constant and its derivative is **Zero**.

Therefore, we can safely remove it from the optimization problem.

---

The new objective function becomes:

$$
\tilde{\mathcal{L}}^{(t)} = \sum_{i=1}^{n} \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \Omega(f_t)
$$

This is the function we will now differentiate to find the optimal leaf weights.

 [Reference:  Gradient Tree Boosting (Section 2.2)](https://arxiv.org/pdf/1603.02754)

-----

Now we focus on the fourth term:

$$
\Omega(f_t)
$$

The regularization term in XGBoost is defined as:

$$
\Omega(f_t) = \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2
$$

Where:

- $T$ = number of leaf nodes in the tree  
- $\gamma$ = penalty for adding a new leaf  
- $\lambda$ = L2 regularization parameter  
- $w_j$ = weight of the $j$-th leaf  

This term penalizes:
- large number of leaves (complex trees)  
- large leaf weights  

and helps control overfitting.

---
Substituting the regularization term into the objective function:

$$
\tilde{\mathcal{L}}^{(t)} = \sum_{i=1}^{n} \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2
$$

For now, assume that the tree has $T$ leaf nodes.

Each data point $x_i$ falls into one of these leaf nodes, and the prediction of the tree is the weight of that leaf.

So we can rewrite:

$$
f_t(x_i) = w_j \quad \text{where } x_i \text{ belongs to leaf } j
$$

Using this, the objective function becomes:

$$
\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^{T} \left[ \sum_{i \in I_j} g_i w_j + \frac{1}{2} \sum_{i \in I_j} h_i w_j^2 \right] + \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2
$$

Where:
- $I_j$ is the set of data points in leaf $j$
- $w_j$ is the weight of leaf $j$

> how the last tranformation happen we understand by an example [below]

----
----

Starting from:

$$
\tilde{\mathcal{L}}^{(t)} = \sum_{i=1}^{n} \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2
$$

Since each data point $x_i$ belongs to a leaf $j$, we can group terms by leaf nodes.

Rewriting the summation:

$$
\sum_{i=1}^{n} \;\; \longrightarrow \;\; \sum_{j=1}^{T} \sum_{i \in I_j}
$$

So the objective becomes:

$$
\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^{T} \left[ \sum_{i \in I_j} g_i w_j + \frac{1}{2} \sum_{i \in I_j} h_i w_j^2 \right] + \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2
$$

---

Expanding the inner terms (intuition):

$$
g_1 f_t(x_1) + \frac{1}{2} h_1 f_t^2(x_1) + g_2 f_t(x_2) + \frac{1}{2} h_2 f_t^2(x_2) + \dots + g_n f_t(x_n) + \frac{1}{2} h_n f_t^2(x_n)
$$

Since all data points in the same leaf share the same value $w_j$, we can factor it out inside each leaf:

$$
= \sum_{j=1}^{T} \left[ w_j \sum_{i \in I_j} g_i + \frac{1}{2} w_j^2 \sum_{i \in I_j} h_i \right]
$$

---

So the final form becomes:

$$
\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^{T} \left[ w_j \sum_{i \in I_j} g_i + \frac{1}{2} w_j^2 \sum_{i \in I_j} h_i \right] + \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2
$$

This form is important because now the objective is expressed directly in terms of leaf weights $w_j$, which we can optimize.

> Understanding by Example 

To understand the notation $i \in I_j$, let’s use the example.

We have a decision tree split:

- If CGPA < 7 → Leaf 1 with weight $w_1 = 5$  
- If CGPA ≥ 7 → Leaf 2 with weight $w_2 = 10$  

Data:

| i | CGPA | Package |
|---|------|---------|
| 1 | 7.1  | 4.5     |
| 2 | 8.2  | 6.9     |
| 3 | 6.5  | 5.1     |
| 4 | 9.1  | 10      |

---

### What does $i \in I_j$ mean?

$I_j$ represents the set of data points that fall into leaf $j$.

From the tree:

- Leaf 1 (CGPA < 7):  
  $$
  I_1 = \{3\}
  $$

- Leaf 2 (CGPA ≥ 7):  
  $$
  I_2 = \{1, 2, 4\}
  $$

So, $i \in I_j$ simply means:

"All data points that belong to leaf $j$"

Yes, we are doing the same summation as before, but now we are **grouping terms based on tree leaves**.

---

### Why is $f_t(x_i) = w_j$?

A decision tree does not output a continuous function like linear regression.

Instead, it outputs a **constant value per leaf**.

So:

- If $x_i$ falls in leaf 1 → output is $w_1$
- If $x_i$ falls in leaf 2 → output is $w_2$

Mathematically:

$$
f_t(x_i) = w_j \quad \text{if } x_i \in I_j
$$

---

### Expanding with the example

Original term:

$$
g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i)
$$

Now substitute $f_t(x_i)$:

- For $i = 3$ (Leaf 1):

$$
g_3 w_1 + \frac{1}{2} h_3 w_1^2
$$

- For $i = 1, 2, 4$ (Leaf 2):

$$
g_1 w_2 + \frac{1}{2} h_1 w_2^2
+ g_2 w_2 + \frac{1}{2} h_2 w_2^2
+ g_4 w_2 + \frac{1}{2} h_4 w_2^2
$$

---

### Grouping by leaf

Leaf 1:

$$
\sum_{i \in I_1} g_i w_1 + \frac{1}{2} \sum_{i \in I_1} h_i w_1^2
$$

Leaf 2:

$$
\sum_{i \in I_2} g_i w_2 + \frac{1}{2} \sum_{i \in I_2} h_i w_2^2
$$

---

### Final Insight

- $f_t(x_i)$ is the **output of the decision tree**
- The tree outputs **constant values per leaf**
- These constants are the **leaf weights $w_j$**

So:

$$
f_t(x_i) = w_j
$$

and

$$
i \in I_j
$$

means we are grouping all data points that fall into the same leaf and assigning them the same output.

So the final form becomes:

$$
\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^{T} \left[ w_j \sum_{i \in I_j} g_i + \frac{1}{2} w_j^2 \sum_{i \in I_j} h_i \right] + \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2
$$

---

Combining the quadratic terms:

$$
\frac{1}{2} w_j^2 \sum_{i \in I_j} h_i + \frac{1}{2} \lambda w_j^2
= \frac{1}{2} \left( \sum_{i \in I_j} h_i + \lambda \right) w_j^2
$$

---

So the objective function simplifies to:

$$
\tilde{\mathcal{L}}^{(t)} =
\sum_{j=1}^{T}
\left[
w_j \sum_{i \in I_j} g_i
+
\frac{1}{2}
\left( \sum_{i \in I_j} h_i + \lambda \right) w_j^2
\right]
+ \gamma T
$$

This is the final form of the objective function expressed in terms of leaf weights $w_j$.

-----
-----

Now we differentiate the objective function with respect to $w_j$:

$$
\tilde{\mathcal{L}}^{(t)} =
\sum_{j=1}^{T}
\left[
w_j \sum_{i \in I_j} g_i
+
\frac{1}{2}
\left( \sum_{i \in I_j} h_i + \lambda \right) w_j^2
\right]
+ \gamma T
$$

---

Taking derivative with respect to $w_j$:

$$
\frac{\partial \tilde{\mathcal{L}}^{(t)}}{\partial w_j}
=
\sum_{i \in I_j} g_i
+
\left( \sum_{i \in I_j} h_i + \lambda \right) w_j
$$

---

Set derivative to zero for minimization:

$$
\sum_{i \in I_j} g_i
+
\left( \sum_{i \in I_j} h_i + \lambda \right) w_j
= 0
$$

---

Solving for $w_j$:

$$
w_j =
-\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}
$$

---

This is the optimal leaf weight for leaf $j$ in XGBoost.

---
---

# Output value for regression 



The optimal leaf weight is given by:

$$
w_j =
-\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}
$$

Where:

- $g_i$ = gradient (first derivative of loss)  
- $h_i$ = Hessian (second derivative of loss)  

So:

$$
\text{numerator} \rightarrow \text{sum of gradients}
$$

$$
\text{denominator} \rightarrow \text{sum of Hessians} + \lambda
$$

This is the output value assigned to each leaf in XGBoost for regression.

The gradient and Hessian are defined as:

$$
g_i = \frac{\partial L\left(y_i, \hat{y}_i^{(t-1)}\right)}{\partial \hat{y}_i^{(t-1)}}
$$

$$
h_i = \frac{\partial^2 L\left(y_i, \hat{y}_i^{(t-1)}\right)}{\partial \left(\hat{y}_i^{(t-1)}\right)^2}
$$

---

These depend on the choice of loss function:

- For regression → Mean Squared Error  
- For classification → Log loss (log-odds)  

So, $g_i$ and $h_i$ change depending on the problem we are solving.

For regression, we use Mean Squared Error (MSE) as the loss function:

$$
L = \frac{1}{2} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

---

The gradient is computed with respect to the **previous prediction**:

$$
g_i = \frac{\partial L\left(y_i, \hat{y}_i^{(t-1)}\right)}{\partial \hat{y}_i^{(t-1)}}
$$

$$
= \frac{\partial}{\partial \hat{y}_i^{(t-1)}} \left( \frac{1}{2} (y_i - \hat{y}_i^{(t-1)})^2 \right)
$$

$$
= \hat{y}_i^{(t-1)} - y_i
$$

---

So, the gradient becomes:

$$
g_i = \hat{y}_i^{(t-1)} - y_i
$$

which represents the error of the previous model.

We know:

$$
g_i = \hat{y}_i^{(t-1)} - y_i
$$

This can be rewritten as:

$$
g_i = - (y_i - \hat{y}_i^{(t-1)})
$$

---

But:

$$
\text{residual}_i = y_i - \hat{y}_i^{(t-1)}
$$

So:

$$
g_i = - \text{residual}_i
$$

---

Substituting into the leaf weight formula:

$$
w_j =
-\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}
$$

$$
=
-\frac{\sum_{i \in I_j} (-\text{residual}_i)}{\sum_{i \in I_j} h_i + \lambda}
$$

$$
=
\frac{\sum_{i \in I_j} \text{residual}_i}{\sum_{i \in I_j} h_i + \lambda}
$$

---

So, the numerator becomes:

$$
\sum \text{residuals}
$$

which connects XGBoost directly to the idea of fitting residuals in gradient boosting.

Now we compute the Hessian:

$$
h_i = \frac{\partial^2 L}{\partial \left(\hat{y}_i^{(t-1)}\right)^2}
$$

For MSE:

$$
L = \frac{1}{2}(y_i - \hat{y}_i^{(t-1)})^2
$$

First derivative:

$$
g_i = \hat{y}_i^{(t-1)} - y_i
$$

Second derivative:

$$
h_i = \frac{\partial}{\partial \hat{y}_i^{(t-1)}} (\hat{y}_i^{(t-1)} - y_i) = 1
$$

---

So:

$$
h_i = 1
$$

---

Now summing over a leaf:

$$
\sum_{i \in I_j} h_i = \sum_{i \in I_j} 1 = |I_j|
$$

which is simply the **number of data points (residuals) in that leaf**.

---

Substituting into the leaf weight formula:

$$
w_j =
\frac{\sum_{i \in I_j} \text{residual}_i}{\sum_{i \in I_j} h_i + \lambda}
$$

$$
=
\frac{\sum_{i \in I_j} \text{residual}_i}{|I_j| + \lambda}
$$

---

Final result:

$$
w_j =
\frac{\sum \text{residuals}}{\text{number of residuals} + \lambda}
$$

This is exactly the output value formula for XGBoost in regression.

# Output value for Classification 

The optimal leaf weight remains:

$$
w_j =
-\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}
$$

Where:

$$
g_i = \frac{\partial L\left(y_i, \hat{y}_i^{(t-1)}\right)}{\partial \hat{y}_i^{(t-1)}}
\quad , \quad
h_i = \frac{\partial^2 L\left(y_i, \hat{y}_i^{(t-1)}\right)}{\partial \left(\hat{y}_i^{(t-1)}\right)^2}
$$

---

For classification, we use **log loss (binary cross-entropy)**.

From gradient boosting (already derived):

$$
g_i = \hat{y}_i^{(t-1)} - y_i
$$

$$
h_i = \hat{y}_i^{(t-1)} (1 - \hat{y}_i^{(t-1)})
$$

---

Substituting into the formula:

$$
w_j =
\frac{\sum_{i \in I_j} (y_i - \hat{y}_i^{(t-1)})}
{\sum_{i \in I_j} \hat{y}_i^{(t-1)} (1 - \hat{y}_i^{(t-1)}) + \lambda}
$$

---

So for classification:

- Numerator → sum of residuals  
- Denominator → sum of $P(1 - P)$ + $\lambda$  

This matches the XGBoost output value formula for classification.

For classification, the leaf output becomes:

$$
w_j =
\frac{\sum \text{residuals}}{\sum_{i \in I_j} P_i (1 - P_i) + \lambda}
$$

Where:

- $P_i$ = predicted probability for data point $i$  
- $\text{residual}_i = y_i - P_i$  

---

So:

- Numerator → sum of residuals  
- Denominator → sum of $P_i (1 - P_i)$ + $\lambda$  

This is the final output value formula for XGBoost in classification.

# Derivation of Similarity Score 



We start with the objective function:

$$
\tilde{\mathcal{L}}^{(t)} =
\sum_{j=1}^{T}
\left[
\left( \sum_{i \in I_j} g_i \right) w_j
+
\frac{1}{2}
\left( \sum_{i \in I_j} h_i + \lambda \right) w_j^2
\right]
+ \gamma T
$$

---

We already derived the optimal leaf weight:

$$
w_j =
-\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}
$$

---

Now, we substitute $w_j$ back into the objective function to compute the score of each leaf.

This value represents how good a split is, and is called the **Similarity Score** in XGBoost.

Substituting the optimal value of $w_j$ into the objective function:

$$
w_j =
-\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}
$$

---

The objective becomes:

$$
\tilde{\mathcal{L}}^{(t)} =
\sum_{j=1}^{T}
\left[
\left( \sum_{i \in I_j} g_i \right)
\left(
-\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}
\right)
+
\frac{1}{2}
\left( \sum_{i \in I_j} h_i + \lambda \right)
\left(
\frac{\left( \sum_{i \in I_j} g_i \right)^2}{\left( \sum_{i \in I_j} h_i + \lambda \right)^2}
\right)
\right]
+ \gamma T
$$

---

Simplifying each term:

First term:

$$
-\frac{\left( \sum_{i \in I_j} g_i \right)^2}{\sum_{i \in I_j} h_i + \lambda}
$$

Second term:

$$
\frac{1}{2}
\cdot
\frac{\left( \sum_{i \in I_j} g_i \right)^2}{\sum_{i \in I_j} h_i + \lambda}
$$

---

Combining both terms:

$$
-\frac{\left( \sum g_i \right)^2}{\sum h_i + \lambda}
+
\frac{1}{2}
\frac{\left( \sum g_i \right)^2}{\sum h_i + \lambda}
=
-\frac{1}{2}
\frac{\left( \sum g_i \right)^2}{\sum h_i + \lambda}
$$

---

So the objective becomes:

$$
\tilde{\mathcal{L}}^{(t)} =
-\frac{1}{2}
\sum_{j=1}^{T}
\frac{\left( \sum_{i \in I_j} g_i \right)^2}{\sum_{i \in I_j} h_i + \lambda}
+ \gamma T
$$

This is the expression used to compute the **similarity score**.

The optimal value of the objective function after substitution becomes:

$$
\tilde{\mathcal{L}}^{(t)}(q) =
-\frac{1}{2}
\sum_{j=1}^{T}
\frac{\left( \sum_{i \in I_j} g_i \right)^2}{\sum_{i \in I_j} h_i + \lambda}
+ \gamma T
$$

---

This expression can be used as a **scoring function** to measure the quality of a tree structure $q$.

It is analogous to impurity measures (like entropy or Gini) in decision trees, but here it is derived from a general objective function.

---

For a single leaf, the **Similarity Score** is defined as:

$$
\text{Similarity Score} =
\frac{\left( \sum g_i \right)^2}{\sum h_i + \lambda}
$$

This score helps determine how good a split is in XGBoost.

## Final Calculation of Similarity Score

From the optimized objective function:

$$
\tilde{\mathcal{L}}^{(t)} =
-\frac{1}{2}
\sum_{j=1}^{T}
\frac{\left( \sum_{i \in I_j} g_i \right)^2}{\sum_{i \in I_j} h_i + \lambda}
+ \gamma T
$$

The term inside the summation represents the contribution of each leaf.

---

### Similarity Score (per leaf)

$$
\text{Similarity Score} =
\frac{\left( \sum_{i \in I_j} g_i \right)^2}{\sum_{i \in I_j} h_i + \lambda}
$$

---

## Regression Case

We know:

$$
g_i = -\text{residual}_i
\quad , \quad
h_i = 1
$$

So:

$$
\sum_{i \in I_j} h_i = N
$$

Thus, the similarity score becomes:

$$
\text{Similarity Score} =
\frac{\left( \sum \text{residuals} \right)^2}{N + \lambda}
$$

---

## Classification Case

We know:

$$
g_i = -\text{residual}_i
\quad , \quad
h_i = P_i (1 - P_i)
$$

So:

$$
\sum_{i \in I_j} h_i = \sum_{i \in I_j} P_i (1 - P_i)
$$

Thus, the similarity score becomes:

$$
\text{Similarity Score} =
\frac{\left( \sum \text{residuals} \right)^2}{\sum P_i (1 - P_i) + \lambda}
$$

---

This is the final formula used in XGBoost to evaluate splits.

---
---

When we consider a single node (parent), it is treated as a tree with only one leaf.

So:

$$
T = 1
$$

---

### Objective for Parent Node

$$
\tilde{\mathcal{L}}^{(t)}(q_{\text{parent}}) =
-\frac{1}{2}
\frac{\left( \sum_{i \in I} g_i \right)^2}{\sum_{i \in I} h_i + \lambda}
+ \gamma
$$

Here:
- $I$ = all data points in the parent node  
- $T = 1 \Rightarrow \gamma T = \gamma$

---

### After Split (Two Leaf Nodes)

Now the node is split into left and right:

- Left leaf: $I_L$
- Right leaf: $I_R$

So:

$$
T = 2
$$

---

### Objective for Child Nodes

$$
\tilde{\mathcal{L}}^{(t)}(q_{\text{children}}) =
-\frac{1}{2}
\left[
\frac{\left( \sum_{i \in I_L} g_i \right)^2}{\sum_{i \in I_L} h_i + \lambda}
+
\frac{\left( \sum_{i \in I_R} g_i \right)^2}{\sum_{i \in I_R} h_i + \lambda}
\right]
+ 2\gamma
$$

---

### Key Insight

- Parent node → $T = 1$
- After split → $T = 2$
- Regularization increases from $\gamma$ to $2\gamma$

This is why splitting a node is only beneficial if the gain from splitting outweighs the increase in regularization.

----
----
We compute the gain (loss reduction) from splitting as:

$$
\mathcal{L}_{\text{split}} =
\tilde{\mathcal{L}}^{(t)}(q_{\text{parent}})
-
\tilde{\mathcal{L}}^{(t)}(q_{\text{children}})
$$

---

After simplification, the loss reduction becomes:

$$
\mathcal{L}_{\text{split}} =
\frac{1}{2}
\left[
\frac{\left( \sum_{i \in I_L} g_i \right)^2}{\sum_{i \in I_L} h_i + \lambda}
+
\frac{\left( \sum_{i \in I_R} g_i \right)^2}{\sum_{i \in I_R} h_i + \lambda}
-
\frac{\left( \sum_{i \in I} g_i \right)^2}{\sum_{i \in I} h_i + \lambda}
\right]
- \gamma
$$

---

Where:

- $I_L$ = set of data points in left child  
- $I_R$ = set of data points in right child  
- $I = I_L \cup I_R$ (parent node)

---

This formula is used in practice to evaluate split candidates in XGBoost.

A split is accepted only if:

$$
\mathcal{L}_{\text{split}} > 0
$$